**Laboratorio de Métodos Cuantitativos Aplicados a la Gestión**

---

# **Clase 07 - Modelado de problemas organizacionales mediante sistemas de ecuaciones lineales**

## Complemento

Este notebook se complementa con la presentación: **AlgebrayORG.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

## ¿Qué vamos a hacer en esta clase?

La clase pasada aprendimos a operar con matrices. Hoy las usamos para lo que fueron inventadas:
**resolver varios problemas entrelazados al mismo tiempo**.

| Parte | Tema | Herramienta |
|---|---|---|
| **A** | De un problema de gestión a un sistema de ecuaciones | Lápiz y papel |
| **B** | Resolverlo en Python | `np.linalg.solve` |
| **C** | Cuándo el sistema **no** tiene solución | El determinante |
| **D** | Un caso más grande: planificación de producción | Sistema 3×3 |
| **E** | Leontief: el sistema de ecuaciones más famoso de la economía | Matriz insumo-producto |
| **F** | Datos reales: la matriz del INDEC 1997 | 124 sectores de la economía argentina |

> **La idea de fondo:** cuando una decisión depende de otra, que depende de otra, no alcanza con
> despejar de a una. Hay que resolverlas todas juntas. Eso es un sistema de ecuaciones.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)

# RAMA: de qué versión del repositorio se leen los datos.
# Si algún archivo te da error 404, probá con RAMA = "juan"
RAMA = "main"
URL = f"https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/{RAMA}/DF/"

---
# 🏭 Parte A — De un problema de gestión a un sistema

**El caso.** Una fábrica produce **sillas** y **mesas**. Cada una consume horas de dos sectores:

| | Carpintería | Pintura |
|---|---|---|
| **Una silla** | 2 hs | 1 h |
| **Una mesa** | 4 hs | 3 hs |
| **Disponible por semana** | **220 hs** | **150 hs** |

**La pregunta del gerente:** ¿cuántas sillas y cuántas mesas hay que producir para usar
*exactamente* toda la capacidad instalada, sin que sobren ni falten horas?

---

**Paso 1 — Nombrar las incógnitas.**

$$s = \text{cantidad de sillas} \qquad m = \text{cantidad de mesas}$$

**Paso 2 — Escribir una ecuación por cada restricción.**

$$\begin{cases} 2s + 4m = 220 & \text{(horas de carpintería)} \\ 1s + 3m = 150 & \text{(horas de pintura)} \end{cases}$$

**Paso 3 — Reescribirlo como matrices.** Esta es la traducción clave:

$$\underbrace{\begin{pmatrix} 2 & 4 \\ 1 & 3 \end{pmatrix}}_{A \ \text{(tecnología)}} \cdot \underbrace{\begin{pmatrix} s \\ m \end{pmatrix}}_{x \ \text{(incógnitas)}} = \underbrace{\begin{pmatrix} 220 \\ 150 \end{pmatrix}}_{b \ \text{(recursos)}}$$

En forma compacta: $$A \cdot x = b$$

- **A** describe *cómo* produce la empresa (la tecnología). Cada **fila** es un recurso, cada **columna** un producto.
- **b** es *cuánto* tiene disponible.
- **x** es lo que queremos averiguar.

---
# 🐍 Parte B — Resolverlo en Python

Si esto fuera álgebra común, despejaríamos $x = b / A$. Con matrices la división no existe, pero
existe algo equivalente: multiplicar por la **matriz inversa**.

$$A \cdot x = b \quad \Longrightarrow \quad x = A^{-1} \cdot b$$

En Python no hace falta ni calcular la inversa: `np.linalg.solve` lo hace de una y es más preciso.

In [ ]:
A = np.array([[2, 4],      # fila 1: carpintería  → 2 hs por silla, 4 hs por mesa
              [1, 3]])     # fila 2: pintura      → 1 h por silla,  3 hs por mesa

b = np.array([220, 150])   # horas disponibles de cada sector

x = np.linalg.solve(A, b)  # resolver el sistema A·x = b

print("Sillas:", round(x[0], 2))
print("Mesas: ", round(x[1], 2))

**Nunca confíes en un resultado sin verificarlo.** Volvemos a multiplicar `A · x` y tiene que dar `b`:

In [ ]:
verificacion = A @ x     # el símbolo @ multiplica matrices (NO uses * , eso multiplica elemento a elemento)

print("A · x =", verificacion)
print("b     =", b)
print("¿Coinciden?", np.allclose(verificacion, b))   # allclose: compara tolerando errores mínimos de redondeo

> ⚠️ **`@` vs `*`** es el error más común de la clase.
> - `A @ x` → **producto matricial** (el del álgebra).
> - `A * x` → multiplica elemento por elemento. No da error, **da otro número**. Peligrosísimo.

### Interpretación económica

Producir **90 sillas y 20 mesas** agota exactamente las 220 horas de carpintería y las 150 de pintura.
No sobra capacidad ociosa ni hace falta pagar horas extra.

Ese "exactamente" es lo que distingue un **sistema de ecuaciones** (buscamos igualdad) de la
**programación lineal** que vieron con Rita (donde hay *desigualdades*: usar *hasta* 220 horas, y
además maximizar la ganancia).

---
# ⚠️ Parte C — Cuándo el sistema no tiene solución

No todo sistema se puede resolver. El **determinante** de A avisa antes de intentarlo:

| Determinante | Qué significa | Interpretación en la fábrica |
|---|---|---|
| **≠ 0** | Solución única ✅ | Los sectores aportan información distinta |
| **= 0** | Ninguna solución, o infinitas ❌ | Una ecuación es "redundante": no aporta nada nuevo |

In [ ]:
print("Determinante de A:", np.linalg.det(A))   # det ≠ 0 → tiene solución única

In [ ]:
# Un caso patológico: la segunda máquina es exactamente el doble de la primera
A_mala = np.array([[2, 4],
                   [4, 8]])     # fila 2 = fila 1 × 2  → no aporta información nueva

print("Determinante:", np.linalg.det(A_mala))

try:
    np.linalg.solve(A_mala, np.array([220, 150]))
except np.linalg.LinAlgError as e:
    print("❌ Python no puede resolverlo:", e)

**Traducción al mundo real:** si el determinante da 0, el modelo está mal planteado. Alguien cargó
dos veces la misma restricción, o dos sectores son en realidad el mismo. **Es un error de datos, no de Python.**

---
# 📦 Parte D — Un caso más grande: planificación de producción

Una empresa de electrodomésticos fabrica tres productos y quiere agotar tres recursos escasos.

| Recurso | Producto A | Producto B | Producto C | Disponible |
|---|---|---|---|---|
| Horas de máquina | 3 | 2 | 4 | 2.400 |
| Horas de mano de obra | 2 | 5 | 3 | 2.300 |
| Kg de materia prima | 4 | 3 | 2 | 2.200 |

El planteo es idéntico, solo que ahora la matriz es 3×3.

In [ ]:
A3 = np.array([[3, 2, 4],     # horas de máquina por unidad de A, B y C
               [2, 5, 3],     # horas de mano de obra
               [4, 3, 2]])    # kg de materia prima

b3 = np.array([2400, 2300, 2200])

print("Determinante:", round(np.linalg.det(A3), 2))   # ≠ 0 → adelante

x3 = np.linalg.solve(A3, b3)

for producto, cantidad in zip(["A", "B", "C"], x3):
    print(f"Producto {producto}: {cantidad:,.1f} unidades")

In [ ]:
# Presentamos el resultado como una tabla, que es como se lo mandás al gerente
plan = pd.DataFrame({
    "Producto": ["A", "B", "C"],
    "Unidades": x3.round(1)
})
plan["Participación %"] = (plan["Unidades"] / plan["Unidades"].sum() * 100).round(1)

plan

> 💡 **Cuidado con las soluciones negativas.** Si alguna cantidad hubiera dado negativa, el sistema
> tendría solución *matemática* pero no *económica*: no se pueden producir −40 heladeras. Ahí es donde
> entra la programación lineal, que impone que las cantidades sean ≥ 0.

---
# 🔗 Parte E — Leontief: el sistema de ecuaciones más famoso de la economía

Wassily Leontief ganó el Nobel en 1973 por una idea simple y potente:
**los sectores de una economía se compran insumos entre sí**.

Para producir pan hace falta harina; para producir harina hace falta trigo; para producir trigo hace
falta combustible... y la industria del combustible, a su vez, compra pan para sus empleados.
Todo depende de todo. **Eso es un sistema de ecuaciones.**

---

### El modelo

Cada sector produce una cantidad $x_i$ que se reparte en dos destinos:

$$x_i = \underbrace{\sum_j a_{ij} x_j}_{\text{lo que consumen otros sectores}} + \underbrace{d_i}_{\text{demanda final}}$$

donde $a_{ij}$ = cuánto insumo del sector $i$ hace falta para producir **un peso** del sector $j$.

En forma matricial:

$$x = A x + d \quad \Longrightarrow \quad x - Ax = d \quad \Longrightarrow \quad (I - A)\,x = d$$

Y la solución:

$$\boxed{x = (I - A)^{-1} \, d}$$

A la matriz $(I-A)^{-1}$ se la llama **matriz de Leontief** o *matriz inversa de Leontief*.

### Un ejemplo chico que se puede verificar a mano

Tres sectores: **Agro**, **Industria** y **Servicios**. La matriz $A$ dice, por columna,
cuánto insumo necesita cada sector para producir 1 peso:

In [ ]:
sectores = ["Agro", "Industria", "Servicios"]

# Columna j = qué necesita el sector j. Ej: producir $1 de Industria consume
# $0.30 de Agro, $0.20 de Industria y $0.20 de Servicios.
A_leontief = np.array([
    [0.20, 0.30, 0.10],   # insumos que aporta el Agro
    [0.40, 0.20, 0.30],   # insumos que aporta la Industria
    [0.10, 0.20, 0.20],   # insumos que aporta Servicios
])

pd.DataFrame(A_leontief, index=sectores, columns=sectores)

In [ ]:
# Demanda final: lo que consumen las familias, el Estado y las exportaciones
d = np.array([100, 200, 150])

I = np.eye(3)                          # eye(3): matriz identidad de 3×3
x = np.linalg.solve(I - A_leontief, d)  # resolver (I - A)·x = d

resultado = pd.DataFrame({
    "Demanda final": d,
    "Producción necesaria": x.round(1),
    "Uso intermedio": (x - d).round(1)   # la diferencia se la consumen los propios sectores
}, index=sectores)

resultado

Mirá la última columna: para entregarle 100 al consumidor final, el Agro tiene que producir mucho más,
porque la Industria y los Servicios le compran una parte. **Esa es la magia del modelo:** captura los
efectos indirectos que a ojo se pierden.

### Multiplicadores: ¿qué sector conviene estimular?

Si el gobierno quiere inyectar 1 peso de demanda, ¿en qué sector genera más actividad total?
La respuesta es la **suma de cada columna** de la matriz de Leontief.

In [ ]:
L = np.linalg.inv(I - A_leontief)      # matriz de Leontief (I-A)⁻¹
multiplicadores = L.sum(axis=0)         # axis=0 suma hacia abajo → un valor por columna

for sector, m in zip(sectores, multiplicadores):
    print(f"{sector:12s}: {m:.3f}")

print("\nLeer así: $1 de demanda extra en Industria genera "
      f"${multiplicadores[1]:.2f} de producción total en toda la economía.")

---
# 🇦🇷 Parte F — Datos reales: la matriz del INDEC 1997

Hasta acá, números inventados. Ahora lo hacemos con la **Matriz Insumo-Producto de la economía argentina**,
publicada por el INDEC: **124 sectores**, desde "Cultivo de cereales" hasta "Servicio doméstico".

In [ ]:
# Matriz Z: cuánto le compró cada sector a cada otro sector, en miles de pesos de 1997
Z = pd.read_csv(URL + "MIP_INDEC_1997_MATRIZ_Z.csv", index_col=0)

print("Dimensiones:", Z.shape)
Z.iloc[:5, :3]   # una esquinita, porque son 124×124

### ⚠️ Un problema de datos que hay que resolver primero

Para calcular los coeficientes técnicos necesitamos dividir cada compra por la **producción total** del sector:

$$a_{ij} = \frac{z_{ij}}{x_j}$$

La tentación es usar la suma de la columna de $Z$ como producción total. **Es un error**, y grave.
La suma de la columna son solo los **insumos intermedios**; falta el **valor agregado** (sueldos y ganancias).

Veamos qué pasa si caemos en la trampa:

In [ ]:
consumo_intermedio = Z.values.sum(axis=0)          # suma de cada columna

A_mal = Z.values / consumo_intermedio               # ⚠️ usando el denominador equivocado

print("Suma de las primeras 5 columnas de A:", A_mal.sum(axis=0)[:5].round(4))
print("Determinante de (I - A):", np.linalg.det(np.eye(124) - A_mal))

**Todas las columnas suman exactamente 1**, y el determinante da **0**. No es casualidad: al dividir
una columna por su propia suma, obligamos a que sume 1. Eso hace que $(I-A)$ sea **singular** y que la
inversa explote a números sin sentido (del orden de $10^{16}$).

**En criollo:** el modelo estaría diciendo que la economía no genera ningún valor agregado — que todo lo
que se produce se lo comen los propios sectores como insumos. Un país donde nadie cobra un sueldo.

> 🎯 **La lección de la clase.** Python no te avisa cuando el error es *conceptual*. Te devuelve un número
> igual. El determinante fue el que nos salvó: si da 0, algo está mal planteado.

### La corrección

La producción total es:

$$x_j = \underbrace{\text{consumo intermedio}_j}_{\text{suma de la columna de } Z} + \underbrace{\text{valor agregado}_j}_{\text{sueldos + ganancias}}$$

El valor agregado está en otro cuadro del INDEC, que también tenemos en el repositorio.

In [ ]:
# Valor Agregado Bruto por sector (Cuadro 16 del INDEC, ya limpio en un CSV)
vab_df = pd.read_csv(URL + "MIP_INDEC_1997_VAB.csv")
vab = vab_df["valor_agregado_bruto"].values

X = consumo_intermedio + vab      # ahora sí: producción total = insumos + valor agregado

print("Sectores:", len(X))
print(f"Producción total de la economía: ${X.sum():,.0f} miles de pesos de 1997")
print(f"Participación del valor agregado: {vab.sum() / X.sum():.1%}")

vab_df.head(3)

In [ ]:
A_real = Z.values / X                       # ahora el denominador es el correcto
I124 = np.eye(len(X))

print("Suma de columnas de A → min %.3f | max %.3f | media %.3f"
      % (A_real.sum(axis=0).min(), A_real.sum(axis=0).max(), A_real.sum(axis=0).mean()))
print("Determinante de (I - A): %.6f  → distinto de cero ✅" % np.linalg.det(I124 - A_real))

Ahora las columnas suman entre 0 y 0,92 con una media de 0,49: **la mitad de lo que produce un sector
son insumos y la otra mitad valor agregado**. Eso sí es una economía real.

In [ ]:
L_real = np.linalg.inv(I124 - A_real)                          # matriz de Leontief
mult = pd.Series(L_real.sum(axis=0), index=Z.columns).sort_values(ascending=False)

print("🔝 Los 10 sectores más encadenados de la economía argentina (1997)\n")
print(mult.head(10).round(3).to_string())
print("\n🔻 Los 5 menos encadenados\n")
print(mult.tail(5).round(3).to_string())
print(f"\nMultiplicador promedio: {mult.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

top = mult.head(12).sort_values()
ax.barh(range(len(top)), top.values, color="#243b5e")
ax.set_yticks(range(len(top)))
ax.set_yticklabels([s[:45] for s in top.index], fontsize=9)
ax.axvline(mult.mean(), color="#e07b39", linestyle="--", linewidth=2,
           label=f"Promedio de la economía ({mult.mean():.2f})")
ax.set_xlabel("Multiplicador de producción")
ax.set_title("Sectores con mayor efecto de arrastre — Argentina 1997",
             loc="left", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

### ¿Qué nos dice el gráfico?

Arriba de todo aparece la **agroindustria**: cueros, aceites, carnes, lácteos, alimentos balanceados.
Tiene sentido — son cadenas largas, que compran a muchos otros sectores antes de llegar al producto final.

Abajo quedan los **servicios**: enseñanza pública, actividades inmobiliarias, servicio doméstico.
El servicio doméstico da exactamente **1,00**: no le compra insumos a nadie, así que un peso de demanda
genera un peso de producción y nada más.

> 📌 **Esto es exactamente lo que hace un ministerio de economía** cuando discute a qué sector darle un
> incentivo. Y todo salió de resolver $(I-A)x = d$.

---
# 📝 Ejercicios

**Ejercicio 1.** Una panadería produce pan y facturas. Cada kilo de pan usa 0,5 kg de harina y 10 minutos
de horno; cada kilo de facturas usa 0,4 kg de harina y 20 minutos de horno. Hay 100 kg de harina y
3.000 minutos de horno. Planteá el sistema y resolvelo.

In [ ]:
# Tu respuesta acá

**Ejercicio 2.** Verificá tu resultado del ejercicio 1 con `@` y `np.allclose`. ¿Sobra algún recurso?

In [ ]:
# Tu respuesta acá

**Ejercicio 3.** Con la matriz `A_leontief` de tres sectores, supongamos que la demanda final de Servicios
sube de 150 a 300. ¿Cuánto tiene que aumentar la producción de **cada** sector? ¿Por qué también sube
la del Agro, si nadie pidió más comida?

In [ ]:
# Tu respuesta acá

**Ejercicio 4.** Buscá en `mult` el multiplicador de tres sectores que te interesen
(usá `mult["nombre del sector"]`). ¿Están por encima o por debajo del promedio?

In [ ]:
# Tu respuesta acá

**Ejercicio 5 (integrador).** El gobierno tiene $1.000 millones para inyectar en un solo sector.
Con los datos del INDEC, ¿en cuál conviene ponerlos si el objetivo es maximizar la producción total?
¿Y qué argumento darías **en contra** de decidirlo solamente con este número?

In [ ]:
# Tu respuesta acá

---
## 🧭 Para llevarse

| Concepto | Comando |
|---|---|
| Resolver $Ax = b$ | `np.linalg.solve(A, b)` |
| Producto matricial | `A @ x` (¡no `A * x`!) |
| Verificar la solución | `np.allclose(A @ x, b)` |
| ¿Tiene solución? | `np.linalg.det(A) != 0` |
| Matriz identidad | `np.eye(n)` |
| Matriz inversa | `np.linalg.inv(A)` |
| Leontief | `x = np.linalg.solve(I - A, d)` |

**La moraleja:** el álgebra lineal no es un tema aparte de la gestión. Es la forma de resolver
decisiones que dependen unas de otras — que en una organización son casi todas.